# 05. Test-Time Compute: Process Reward Models (PRMs), Best-of-N Reranking, and MCTS

Test-time compute scaling is the computational engine powering state-of-the-art reasoning architectures (e.g., OpenAI o1/o3, DeepSeek-R1). Instead of relying solely on scaling parameter counts during pretraining, test-time compute dynamically trades inference latency and search depth for reasoning accuracy.

The core paradigm couples dense step-level credit assignment via Process Reward Models (PRMs) with structured tree exploration (MCTS PUCT) and parallel candidate reranking (Best-of-$N$).


## Mathematical Foundations & Governing Equations

### 1. Process Reward Model (PRM) Aggregation Mechanics
Given step correctness probabilities $p_{i, s} \in (0, 1]$ and binary validity mask $m_{i, s} \in \{0, 1\}$ across $S$ reasoning steps:
- **Product Aggregation (Joint Probability)**: $R_{\text{prod}}(i) = \exp\left(\sum_{s=1}^S m_{i, s} \ln(\max(p_{i, s}, \epsilon))\right)$
- **Minimum Aggregation (Weakest Link)**: $R_{\text{min}}(i) = \min_{s: m_{i, s} = 1} p_{i, s}$

### 2. Best-of-$N$ (BoN) Outcome + Process Reranking
$$S_i = \alpha R_{\text{ORM}}(i) + (1 - \alpha) R_{\text{PRM}}(i), \quad i^* = \arg\max_i S_i$$

### 3. MCTS PUCT Action Node Selection
$$U(s, a) = Q(s, a) + c_{\text{puct}} P(s, a) \frac{\sqrt{\sum_b N(s, b)}}{1 + N(s, a)}, \quad a^* = \arg\max_a U(s, a)$$

### 4. Incremental Running-Average Backpropagation
$$N'(s, a^*) = N(s, a^*) + 1, \quad Q'(s, a^*) = Q(s, a^*) + \frac{V - Q(s, a^*)}{N'(s, a^*)}$$


In [ ]:
# Setup, imports, and golden weight loading
import sys
from pathlib import Path
import torch
from safetensors.torch import load_file

repo_root = Path.cwd()
while not (repo_root / "Cargo.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

weights_path = repo_root / "weights" / "08_learning_paradigms_rl_and_alignment" / "03_llm_alignment_and_reasoning" / "05_test_time_compute_prm_best_of_n_and_mcts.safetensors"
assert weights_path.exists(), f"Weights file not found: {weights_path}"
weights = load_file(str(weights_path))
print(f"Loaded {len(weights)} golden tensors on cpu")


In [ ]:
def linear_anchor(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor) -> torch.Tensor:
    """Canonical cross-track parity anchor."""
    return x @ weight + bias


def prm_product_aggregation(step_probs: torch.Tensor, step_mask: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Process Reward Model product aggregation over valid steps."""
    log_probs = torch.log(step_probs.clamp(min=eps)) * step_mask
    return torch.exp(log_probs.sum(dim=-1))


def prm_min_aggregation(step_probs: torch.Tensor, step_mask: torch.Tensor) -> torch.Tensor:
    """Process Reward Model min aggregation (weakest link principle)."""
    masked_probs = torch.where(step_mask > 0.5, step_probs, torch.ones_like(step_probs))
    return masked_probs.min(dim=-1).values


def best_of_n_rerank(
    orm_scores: torch.Tensor,
    prm_scores: torch.Tensor,
    alpha: float = 0.5,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Combines ORM and PRM scores and identifies top candidate."""
    combined = alpha * orm_scores + (1.0 - alpha) * prm_scores
    best_idx = torch.argmax(combined).to(torch.float32).unsqueeze(0)
    return combined, best_idx


def puct_node_selection(
    q_values: torch.Tensor,
    priors: torch.Tensor,
    visit_counts: torch.Tensor,
    c_puct: float = 1.414,
) -> tuple[torch.Tensor, torch.Tensor]:
    """AlphaZero PUCT action node selection."""
    total_visits = visit_counts.sum()
    exploration = c_puct * priors * torch.sqrt(total_visits) / (1.0 + visit_counts)
    puct_scores = q_values + exploration
    selected_action = torch.argmax(puct_scores).to(torch.float32).unsqueeze(0)
    return puct_scores, selected_action


def mcts_backpropagation(
    q_values: torch.Tensor,
    visit_counts: torch.Tensor,
    action: int,
    leaf_value: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Incremental running-average backpropagation update."""
    new_visits = visit_counts.clone()
    new_q = q_values.clone()
    new_visits[action] += 1.0
    new_q[action] += (leaf_value - new_q[action]) / new_visits[action]
    return new_q, new_visits


In [ ]:
# Multi-track numerical parity assertions
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
err_anchor = float(torch.max(torch.abs(y_anchor - weights["test_output"])))
assert err_anchor < 1e-5, f"Anchor parity error: {err_anchor:.2e}"

# 2. PRM product aggregation
prm_prod = prm_product_aggregation(weights["step_probs"], weights["step_mask"])
err_prod = float(torch.max(torch.abs(prm_prod - weights["prm_prod_ref"])))
assert err_prod < 1e-5, f"PRM prod parity error: {err_prod:.2e}"

# 3. PRM min aggregation
prm_min = prm_min_aggregation(weights["step_probs"], weights["step_mask"])
err_min = float(torch.max(torch.abs(prm_min - weights["prm_min_ref"])))
assert err_min < 1e-5, f"PRM min parity error: {err_min:.2e}"

# 4. Best-of-N reranking
comb, b_idx = best_of_n_rerank(weights["orm_scores"], weights["prm_min_ref"], alpha=0.5)
err_comb = float(torch.max(torch.abs(comb - weights["combined_scores_ref"])))
assert err_comb < 1e-5, f"Combined score parity error: {err_comb:.2e}"
err_bidx = float(torch.max(torch.abs(b_idx - weights["best_idx_ref"])))
assert err_bidx < 1e-5, f"Best index parity error: {err_bidx:.2e}"

# 5. PUCT selection
puct, sel_a = puct_node_selection(weights["q_values"], weights["priors"], weights["visit_counts"], c_puct=1.414)
err_puct = float(torch.max(torch.abs(puct - weights["puct_scores_ref"])))
assert err_puct < 1e-5, f"PUCT scores parity error: {err_puct:.2e}"
err_sel = float(torch.max(torch.abs(sel_a - weights["selected_action_ref"])))
assert err_sel < 1e-5, f"Selected action parity error: {err_sel:.2e}"

# 6. MCTS backprop
action = int(weights["selected_action_ref"].item())
q_up, v_up = mcts_backpropagation(weights["q_values"], weights["visit_counts"], action, 0.90)
err_q = float(torch.max(torch.abs(q_up - weights["q_updated_ref"])))
assert err_q < 1e-5, f"Q update parity error: {err_q:.2e}"
err_v = float(torch.max(torch.abs(v_up - weights["visits_updated_ref"])))
assert err_v < 1e-5, f"Visits update parity error: {err_v:.2e}"

print(f"All parity assertions passed successfully (max error {max(err_anchor, err_prod, err_min, err_comb, err_bidx, err_puct, err_sel, err_q, err_v):.2e})")


In [ ]:
from core.benchmark import benchmark_fn, format_benchmark_table

b_prod = benchmark_fn(
    lambda: prm_product_aggregation(weights["step_probs"], weights["step_mask"]),
    iters=100,
    name="PRM Product Aggregation",
)
b_min = benchmark_fn(
    lambda: prm_min_aggregation(weights["step_probs"], weights["step_mask"]),
    iters=100,
    name="PRM Min Aggregation",
)
b_bon = benchmark_fn(
    lambda: best_of_n_rerank(weights["orm_scores"], weights["prm_min_ref"]),
    iters=100,
    name="Best-of-N Combined Rerank",
)
b_puct = benchmark_fn(
    lambda: puct_node_selection(weights["q_values"], weights["priors"], weights["visit_counts"]),
    iters=100,
    name="MCTS PUCT Action Selection",
)

print(format_benchmark_table([b_prod, b_min, b_bon, b_puct]))


In [ ]:
# Failure Mode Contrast: PRM Product Length Penalty vs. Weakest-Link Minimum Aggregation
# Consider two trajectories for a complex math proof:
# Trajectory A: 2-step brief proof with a flawed shortcut: step 1 = 0.90, step 2 = 0.50 (flawed step!)
# Trajectory B: 10-step rigorous, thorough proof: 10 steps each evaluated at 0.92 confidence

steps_a = torch.tensor([[0.90, 0.50]])
mask_a = torch.tensor([[1.0, 1.0]])

steps_b = torch.full((1, 10), 0.92)
mask_b = torch.ones((1, 10))

# Evaluate under Product Aggregation:
prod_a = prm_product_aggregation(steps_a, mask_a).item()
prod_b = prm_product_aggregation(steps_b, mask_b).item()

# Evaluate under Minimum Aggregation:
min_a = prm_min_aggregation(steps_a, mask_a).item()
min_b = prm_min_aggregation(steps_b, mask_b).item()

print(f"Trajectory A (Short flawed shortcut)  - Product: {prod_a:.4f} | Min: {min_a:.4f}")
print(f"Trajectory B (Long rigorous proof)   - Product: {prod_b:.4f} | Min: {min_b:.4f}")

# Under product aggregation, flawed shortcut wins due to length penalty:
assert prod_a > prod_b, "Product aggregation should suffer from length bias penalizing Trajectory B"
# Under minimum aggregation, rigorous proof correctly triumphs:
assert min_b > min_a, "Min aggregation correctly selects rigorous proof over flawed shortcut"

print("Pathological failure mode contrast successfully verified")


## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Dense Step-Level Credit Assignment**: Process Reward Models eliminate reward sparsity by evaluating reasoning soundness at each intermediate step, preventing credit attribution failure in long proofs.
2. **Aggregation Nuance**: While product aggregation reflects joint probability under independence, it severely penalizes chain length. Minimum aggregation ("weakest link principle") robustly filters out logical flaws without penalizing long-horizon derivations.
3. **Exploration via PUCT**: Predictor Upper Confidence bounds for Trees dynamically balance high-probability policy traces with high-value exploratory avenues during search.
4. **Compute Equivalence**: Test-time compute scaling provides an alternative scaling dimension to parameter scale, allowing compact models with search to outperform orders-of-magnitude larger models.

---
### Cross-Domain Generalisation Matrix
- **Mathematical Problem Solving**: PRM and MCTS search explore lemma candidates, backtracking upon encountering incorrect algebra.
- **Autonomous Software Synthesis**: Step verification tests incremental AST modifications against test suites.
- **Biochemical Synthesis Planning**: Tree search navigates plausible retrosynthetic paths using reaction feasibility verifiers.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_test_time_compute_prm_best_of_n_and_mcts.md](../../../markdown/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/05_test_time_compute_prm_best_of_n_and_mcts.md)
- **PyTorch 2.x Module:** [05_test_time_compute_prm_best_of_n_and_mcts.py](../../../pytorch2/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/05_test_time_compute_prm_best_of_n_and_mcts.py)
- **Functional JAX Module:** [05_test_time_compute_prm_best_of_n_and_mcts.py](../../../jax/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/05_test_time_compute_prm_best_of_n_and_mcts.py)
- **Native Rust Candle Track:** [05_test_time_compute_prm_best_of_n_and_mcts.rs](../../../candle/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/05_test_time_compute_prm_best_of_n_and_mcts.rs)
- **Hardware Kernel (CPU / WGSL):** [05_test_time_compute_prm_best_of_n_and_mcts.rs](../../../kernels/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/05_test_time_compute_prm_best_of_n_and_mcts.rs)
- **Golden SafeTensors:** [05_test_time_compute_prm_best_of_n_and_mcts.safetensors](../../../weights/08_learning_paradigms_rl_and_alignment/03_llm_alignment_and_reasoning/05_test_time_compute_prm_best_of_n_and_mcts.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. GRPO DAPO and Verifiable Reward Reasoning](04_grpo_dapo_and_verifiable_reward_reasoning.ipynb) | [Master Index](../../../README.md#08_learning_paradigms_rl_and_alignment)
